# Análise Exploratória — Bank Marketing

Este notebook executa a camada de tradução e validação do M1 a partir do snapshot bruto. O objetivo é compreender qualidade, desbalanceamento, canais e histórico de campanha sem criar transformações exclusivas no notebook.

**Unidade de observação:** registro de contato de uma campanha.  
**Target:** `resultado`, igual a 1 quando houve assinatura do depósito a prazo.  
**Ação histórica:** `canal_contato`, com os braços `celular` e `telefone`.  
**Atenção:** associações históricas não representam efeito causal.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pandas.api.types import is_string_dtype

# Localiza a raiz independentemente de o notebook iniciar na raiz ou em notebooks/.
current_path = Path.cwd().resolve()
project_root = current_path if (current_path / 'configs').exists() else current_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.contracts import TRANSLATED_BUSINESS_COLUMNS
from src.data.translate import build_interim_dataset
from src.data.validate import validate_translated_dataset

data_config_path = project_root / 'configs' / 'data.yaml'
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 30)

## 1. Construção reproduzível da camada PT-BR

A função abaixo verifica o SHA-256 da fonte, valida schema e categorias, traduz os 21 campos, cria `event_id` e grava a camada `interim`.

In [ ]:
# Executa a mesma função usada pela linha de comando e pelos testes.
translated_frame, lineage_metadata = build_interim_dataset(data_config_path)
validation_report = validate_translated_dataset(translated_frame)

pd.DataFrame({
    'indicador': ['registros', 'colunas', 'células nulas', 'duplicatas de negócio'],
    'valor': [
        validation_report.row_count,
        validation_report.column_count,
        validation_report.null_cell_count,
        validation_report.duplicate_business_rows,
    ],
})

In [ ]:
translated_frame.head()

As 12 duplicatas são calculadas sem `event_id`. Elas permanecem nesta camada para preservar paridade com a fonte e serão removidas de forma versionada no M2.

## 2. Distribuição do target

In [ ]:
target_summary = (
    translated_frame['resultado']
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis('resultado')
    .to_frame('registros')
)
target_summary['proporcao'] = target_summary['registros'] / len(translated_frame)
target_summary

In [ ]:
figure, axis = plt.subplots(figsize=(6, 4))
sns.countplot(data=translated_frame, x='resultado', ax=axis)
axis.set(
    title='Distribuição da conversão',
    xlabel='Resultado (0 = não converteu; 1 = converteu)',
    ylabel='Quantidade de registros',
)
plt.show()

A classe positiva representa aproximadamente 11,3% dos registros. Acurácia isolada será inadequada; a modelagem deverá priorizar PR-AUC, calibração e métricas por classe.

## 3. Qualidade semântica e valores desconhecidos

`desconhecido` não é nulo técnico. A categoria representa ausência semântica informada pela fonte e deve ser preservada e monitorada.

In [ ]:
string_columns = [
    column for column in translated_frame.columns
    if is_string_dtype(translated_frame[column].dtype)
]
unknown_summary = pd.DataFrame([
    {
        'coluna': column,
        'quantidade': int(translated_frame[column].eq('desconhecido').sum()),
        'proporcao': translated_frame[column].eq('desconhecido').mean(),
    }
    for column in string_columns
    if translated_frame[column].eq('desconhecido').any()
]).sort_values('proporcao', ascending=False)
unknown_summary

## 4. Perfil numérico

In [ ]:
numeric_columns = translated_frame.select_dtypes(include='number').columns.drop(
    ['event_id', 'resultado']
)
translated_frame[numeric_columns].describe().T

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4))
sns.histplot(data=translated_frame, x='idade', bins=30, ax=axis)
axis.set(title='Distribuição de idade', xlabel='Idade', ylabel='Registros')
plt.show()

`idade` é campo somente de auditoria no MVP. O gráfico descreve a base, mas não autoriza seu uso para escolher o canal.

## 5. Conversão observada por canal

In [ ]:
channel_summary = (
    translated_frame.groupby('canal_contato', observed=True)['resultado']
    .agg(registros='size', conversoes='sum', taxa_conversao='mean')
    .sort_values('taxa_conversao', ascending=False)
)
channel_summary

In [ ]:
figure, axis = plt.subplots(figsize=(6, 4))
sns.barplot(
    data=channel_summary.reset_index(),
    x='canal_contato',
    y='taxa_conversao',
    ax=axis,
)
axis.set(title='Conversão histórica por canal', xlabel='Canal', ylabel='Taxa de conversão')
axis.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
plt.show()

O canal celular possui conversão histórica superior, mas a alocação não foi randomizada. Diferenças de público, período e estratégia podem explicar parte ou todo o resultado.

## 6. Histórico de campanhas

In [ ]:
previous_outcome_summary = (
    translated_frame.groupby('resultado_campanha_anterior', observed=True)['resultado']
    .agg(registros='size', conversoes='sum', taxa_conversao='mean')
    .sort_values('taxa_conversao', ascending=False)
)
previous_outcome_summary

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=previous_outcome_summary.reset_index(),
    x='resultado_campanha_anterior',
    y='taxa_conversao',
    ax=axis,
)
axis.set(
    title='Conversão por resultado da campanha anterior',
    xlabel='Resultado anterior',
    ylabel='Taxa de conversão',
)
axis.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
plt.show()

## 7. Demonstração do vazamento por duração

In [ ]:
duration_summary = translated_frame.groupby('resultado')['duracao_contato'].agg(
    registros='size',
    media='mean',
    mediana='median',
)
duration_summary

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=translated_frame,
    x='resultado',
    y='duracao_contato',
    showfliers=False,
    ax=axis,
)
axis.set(
    title='Duração observada por resultado',
    xlabel='Resultado',
    ylabel='Duração do contato em segundos',
)
plt.show()

A duração média é muito maior entre conversões, mas só é conhecida após o contato. Ela permanece na camada canônica para auditoria desta evidência e está bloqueada no contrato do experimento; o M2 deverá excluí-la das features.

## 8. Fatia de auditoria por profissão

In [ ]:
profession_audit = (
    translated_frame.groupby('profissao', observed=True)['resultado']
    .agg(registros='size', conversoes='sum', taxa_conversao='mean')
    .sort_values('taxa_conversao', ascending=False)
)
profession_audit

A análise por profissão serve para identificar disparidades e possíveis proxies. O contrato do M0 classifica esse campo como somente auditoria, portanto ele não entra diretamente na política do MVP.

## 9. Conclusões e decisões para o M2

- A tradução mantém 41.188 registros, 21 campos de negócio e adiciona um identificador técnico estável.
- Não há nulos técnicos, mas seis campos contêm a categoria `desconhecido`; inadimplência possui a maior incidência.
- Existem 12 duplicatas de negócio, preservadas no M1 e destinadas à remoção controlada no M2.
- A conversão positiva é de aproximadamente 11,3%, confirmando o desbalanceamento.
- Celular apresenta maior conversão histórica, mas a associação não é causal.
- Resultado anterior é informativo e está disponível antes da nova decisão.
- Duração é um vazamento pós-contato e deve ser bloqueada antes do treino.
- Campos demográficos e financeiros permanecem somente para auditoria no MVP.

O M2 deverá remover duplicatas, derivar a sentinela de contato anterior, separar ação/target/contexto e criar splits antes de ajustar qualquer transformação.